# Solar Filament V13 — Exact V7 Recovery + Frozen-Baseline Delta-PQ (GPU)

**A reproducible Kaggle experiment constructed directly from your original V7 notebook and V12.1 logs.**

The only *publicly confirmed* score is **0.34**. V12.1 reported a **+0.0488 grouped out-of-fold local-PQ change over an incorrect reconstructed baseline**, not a verified improvement on the public submission. V13 fixes the historical inference contract first.

**Recommended execution order:** `MODE="RECOVER"` → inspect recovery outputs → `MODE="ACTIONS"` → inspect audit → `MODE="SUBMIT"`. You can set `MODE="ALL"`, but staged runs make diagnosis and Kaggle time limits easier. A GPU T4 is recommended for inference; sklearn action models run on CPU.

**Fail closed**: missing/mismatched hashes are fatal; if baseline recovery, PQ improvement, fold stability, prior-audit regression, or official evaluator parity are unverified, `/kaggle/working/submission.csv` is the **byte-exact original 0.34 CSV**, not a fabricated new result. No test annotations are used.

**Provenance note:** The attached V7 notebook was a `TUNE` run. Its final selected `selection.json` was not included. We therefore recover from its authentic candidate family and verify masks against the user's own original CSV. This verification is not access to hidden test labels.

## 0 · Run controls & exact asset hashes

Attach: official MAGFiLO data, V4 `best.pt` and `split_manifest.json`, and `submission2.csv`. The historical V7 result ZIP (optional) supplies some cache entries and screening results. The notebook uses the original `ultralytics==8.4.175`, `INTER_NEAREST`, `max_det=65`, four 1280-pixel tiles, 12-pixel internal seam flags and V7 selection rules.

In [ ]:
# ====== EDIT ONLY THESE RUN CONTROLS ======
MODE = 'RECOVER'                # RECOVER | ACTIONS | SUBMIT | ALL
DEVICE_REQUEST = 'auto'         # auto | cpu | 0  (Kaggle T4 auto chooses GPU 0)
PREVIOUS_V13_ROOT = None       # e.g. '/kaggle/input/my-v13-output/solar_v13_exact_v7_delta'
PREVIOUS_V7_ROOT = None        # optional extracted historical V7 cache folder
USE_ARCHIVED_V7_ZIP = True     # optional prior V7 result ZIP
ENABLE_MICROTILES = False     # enable only after baseline recovery and ablation
ENABLE_RIDGE_ADD = True
ENABLE_UNET = False           # reserved; no unvalidated U-Net weights are silently used
CONFIRMED_OFFICIAL_PQ_PARITY = False  # must be set True only after literal organizer parity test
ALLOW_EXPLORATORY_ACTIONS_WITHOUT_RECOVERY = True

SEED=2026; H=W=2048; TILE_SIDE=1280; TILE_IMGSZ=1024; GLOBAL_IMGSZ=1024
TILES='FOUR'; RAW_CONF=.10; RAW_IOU=.82; MAX_DET=65
MIN_GAIN_TO_PROMOTE=.015; AUDIT_MIN_DELTA=-.005
BASE_SHA='1f272f96b79b0087b493ee69119573a897282dc1573460a050d53e437ead0124'
MODEL_SHA_EXPECTED='c7bee9a2acf8acab7372e701ab19b07e01891e560afc818961e0a7828c5ee340'
SPLIT_SHA_EXPECTED='f9c906951de50e5245c7411ceba5ebd093eff2368f9719c628dc8d1d7e52361c'
OUT=__import__('pathlib').Path('/kaggle/working/solar_v13_exact_v7_delta')
PREVIOUS_V13_ROOT=__import__('pathlib').Path(PREVIOUS_V13_ROOT) if PREVIOUS_V13_ROOT else None
PREVIOUS_V7_ROOT=__import__('pathlib').Path(PREVIOUS_V7_ROOT) if PREVIOUS_V7_ROOT else None
assert MODE in ('RECOVER','ACTIONS','SUBMIT','ALL')
assert ENABLE_UNET is False, ('V13 recovery/action mode does not silently train a U-Net. '
                              'Separate GPU ADD-proposal development is possible after baseline parity.')

## 1 · Dependencies, deterministic setup and environment provenance

Exact historical inference version is pinned in a fresh Kaggle kernel. **Restart the Kaggle session if you changed the installed Ultralytics version after importing it.** Packages are installed only as needed. No hidden competitions labels are accessed.

In [ ]:
import sys, os, gc, json, hashlib, random, time, shutil, subprocess, importlib.util, platform, zipfile, importlib.metadata
from pathlib import Path
from collections import defaultdict, Counter
from itertools import product
from scipy.optimize import linear_sum_assignment
import numpy as np, pandas as pd, cv2
from tqdm.auto import tqdm

# Ultralytics exact historical wheel, essential for baseline recovery.
try: installed = importlib.metadata.version('ultralytics')
except importlib.metadata.PackageNotFoundError: installed=None
if installed != '8.4.175':
    if 'ultralytics' in sys.modules:
        raise RuntimeError('Restart the Kaggle kernel; Ultralytics was imported before pinning 8.4.175.')
    subprocess.check_call([sys.executable,'-m','pip','install','-q','ultralytics==8.4.175'])
for module,pkg in [('pycocotools','pycocotools'),('skimage','scikit-image'),
                   ('sklearn','scikit-learn'),('joblib','joblib')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
import torch, ultralytics, joblib, sklearn, skimage
from ultralytics import YOLO
from pycocotools import mask as coco
from skimage.morphology import skeletonize
from sklearn.ensemble import HistGradientBoostingClassifier
from IPython.display import display
assert ultralytics.__version__=='8.4.175',ultralytics.__version__
DEVICE=('cpu' if DEVICE_REQUEST=='cpu' else (0 if torch.cuda.is_available() and DEVICE_REQUEST=='auto' else DEVICE_REQUEST))
if DEVICE=='auto':DEVICE='cpu'
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);cv2.setNumThreads(0)
if hasattr(torch,'set_num_threads'):torch.set_num_threads(min(4,os.cpu_count() or 2))
for name in ['recovery','diagnostics','diagnostics/review','diagnostics/v7_selection_traces',
             'models','action_tables','frozen_baseline','global','tiles','microtiles']:
    (OUT/name).mkdir(parents=True,exist_ok=True)
print('Mode:',MODE,'device:',DEVICE,'torch:',torch.__version__,'ultralytics:',ultralytics.__version__)
assert not list(Path('/kaggle/input').rglob('*test*annotation*.json')), 'Unexpected test annotation artifact attached. Remove it.'
ENV={'python':sys.version,'torch':torch.__version__,'ultralytics':ultralytics.__version__,
     'sklearn':sklearn.__version__,'skimage':skimage.__version__,
     'opencv':cv2.__version__,'device':str(DEVICE),'platform':platform.platform(),
     'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
(OUT/'environment_fingerprint.json').write_text(json.dumps(ENV,indent=2,default=str))

## 2 · Discover exact frozen CSV, trained checkpoint, official data, and the original V4 split

No filenames are trusted without SHA checks. The 707 official train observations retain V4's 513/97/97 train/tune/audit grouping, with no acquisition-day overlap.

In [ ]:
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for part in iter(lambda:f.read(1<<20),b''):h.update(part)
    return h.hexdigest()

def exact_file(name, expected):
    candidates=[p for p in Path('/kaggle/input').rglob(name) if p.is_file()]
    matches=[p for p in candidates if sha256(p)==expected]
    if len(matches)!=1:
        raise FileNotFoundError(f'Expected exactly one {name} SHA={expected}; matches={matches}')
    return matches[0]

FROZEN=exact_file('submission2.csv',BASE_SHA)
CKPT=exact_file('best.pt',MODEL_SHA_EXPECTED)
SPLIT_FILE=exact_file('split_manifest.json',SPLIT_SHA_EXPECTED)
SOURCE_V4_ROOT=SPLIT_FILE.parent
CHECKPOINT_OVERRIDE=str(CKPT)
MODEL_SHA=sha256(CKPT);SPLIT_SHA=sha256(SPLIT_FILE)
DATA_ROOT=Path('/kaggle/input/datasets/lighittha/solar3k/MAGFiLO_1.0_Kaggle_2026')
if not (DATA_ROOT/'train'/'MAGFiLO_1.0_Annotations_kaggle2026_train.json').is_file():
    found=list(Path('/kaggle/input').rglob('MAGFiLO_1.0_Annotations_kaggle2026_train.json'))
    if len(found)!=1:raise FileNotFoundError('Attach the official MAGFiLO data; found '+str(found))
    DATA_ROOT=found[0].parent.parent
ANNOT_PATH=DATA_ROOT/'train'/'MAGFiLO_1.0_Annotations_kaggle2026_train.json'
ANNS=json.loads(ANNOT_PATH.read_text())
SPLIT=json.loads(SPLIT_FILE.read_text())
assert set(SPLIT)=={'train','tune','audit'}
TRAIN={p.stem:p for p in (DATA_ROOT/'train'/'train_images').iterdir()
       if p.suffix.lower() in ('.jpg','.jpeg','.png')}
TEST={p.stem:p for p in (DATA_ROOT/'test'/'test_images').iterdir()
      if p.suffix.lower() in ('.jpg','.jpeg','.png')}
assert len(TRAIN)==707 and len(TEST)==180,(len(TRAIN),len(TEST))
assert [len(SPLIT[k]) for k in ('train','tune','audit')]==[513,97,97]
assert set().union(*(set(SPLIT[k]) for k in SPLIT))==set(TRAIN)
for a in SPLIT:
    for b in SPLIT:
        if a!=b:
            assert not (set(SPLIT[a])&set(SPLIT[b]))
            assert not ({s[:8] for s in SPLIT[a]}&{s[:8] for s in SPLIT[b]})
AB=defaultdict(list)
for a in ANNS['annotations']:AB[str(a['image_id'])].append(a)
OBS=defaultdict(list)
for meta in ANNS['images']:
    stem=Path(meta['file_name']).stem
    if stem in TRAIN:OBS[stem].append((str(meta['id']),AB[str(meta['id'])]))
assert set(OBS)==set(TRAIN)
shutil.copy2(FROZEN,OUT/'frozen_baseline'/'submission_034.csv')
print('Frozen:',FROZEN,' / Checkpoint:',CKPT,' / Split:',SPLIT_FILE)
print('Observations:',len(TRAIN),'train;',len(TEST),'test;',sum(map(len,OBS.values())),'annotator versions')

## 3 · Validate the real 0.34 submission (1,217 masks, 177 nonempty test images)

`filament_id` is `<image_stem>_<instance_number>`, never an image stem alone. **Do not replace this CSV unless promotion passes.**

In [ ]:
frozen_df=pd.read_csv(FROZEN,dtype=str,keep_default_na=False)
assert frozen_df.columns.tolist()==['filament_id','segmentation_rle']
assert len(frozen_df)==1217
assert frozen_df['filament_id'].is_unique
assert frozen_df['segmentation_rle'].map(bool).all()
FROZEN_MASKS=defaultdict(list)
for fid,s in frozen_df.itertuples(index=False,name=None):
    stem,sep,suffix=fid.rpartition('_')
    assert sep and suffix.isdigit() and stem in TEST,('Bad filament_id',fid)
    FROZEN_MASKS[stem].append((int(suffix),s))
for stem,records in FROZEN_MASKS.items():
    records.sort()
    assert [j for j,_ in records]==list(range(1,len(records)+1)),('Nonsequential ids',stem)
assert len(FROZEN_MASKS)==177
FROZEN_RLES={s:[s0 for _,s0 in values] for s,values in FROZEN_MASKS.items()}
print('Original 0.34 baseline:',len(frozen_df),'masks in',len(FROZEN_MASKS),'images; SHA:',sha256(FROZEN))

## 4 · Exact original V7 geometry and COCO RLE functions (source-preserving)

The following cell is copied from your attached `Solar_Filament_V7_CPU_Native_Tiles_Instance_Discovery (1).ipynb`. This keeps the precise **`INTER_NEAREST`** mapping and internal seam definition.

In [ ]:
def windows(side=TILE_SIDE, scheme=TILES):
    assert 0<side<=H
    positions=[(0,0),(W-side,0),(0,H-side),(W-side,H-side)]  # x, y
    if scheme=='FIVE':positions.append(((W-side)//2,(H-side)//2))
    if scheme not in ('FOUR','FIVE'):raise ValueError(scheme)
    return sorted(set(positions))
TILE_WINDOWS=windows()
assert all(0<=x<=W-TILE_SIDE and 0<=y<=H-TILE_SIDE for x,y in TILE_WINDOWS)

def to_rle(x):
    if isinstance(x,dict):
        return {'size':[H,W],'counts':x['counts'].encode('ascii') if isinstance(x['counts'],str) else x['counts']}
    return {'size':[H,W],'counts':x.encode('ascii') if isinstance(x,str) else x}
def to_ascii(r):
    c=r['counts'];return c.decode('ascii') if isinstance(c,bytes) else str(c)
def encode(m):return coco.encode(np.asfortranarray(np.asarray(m,dtype=np.uint8)))

def tile_mask_to_full(tile_mask,x,y,side=TILE_SIDE):
    tile_mask=np.asarray(tile_mask,dtype=np.uint8)
    if tile_mask.shape!=(side,side):tile_mask=cv2.resize(tile_mask,(side,side),interpolation=cv2.INTER_NEAREST)
    full=np.zeros((H,W),np.uint8)
    full[y:y+side,x:x+side]=tile_mask
    return full

def has_internal_tile_edge(mask,x,y,margin=12):
    if not margin or not np.any(mask):return False
    ys,xs=np.nonzero(mask)
    return bool((x>0 and xs.min()<margin) or (x+TILE_SIDE<W and xs.max()>=TILE_SIDE-margin) or
                (y>0 and ys.min()<margin) or (y+TILE_SIDE<H and ys.max()>=TILE_SIDE-margin))

# Tests: identity and exact placement near both upper and lower image boundaries
small=np.zeros((TILE_SIDE,TILE_SIDE),np.uint8);small[22:29,55:66]=1
for x,y in TILE_WINDOWS:
    full=tile_mask_to_full(small,x,y)
    assert full.shape==(H,W) and int(full.sum())==int(small.sum())
    assert int(full[y+22:y+29,x+55:x+66].sum())==77
    assert coco.area(encode(full))==77
print('Native tile windows:',TILE_WINDOWS,'coordinate unit tests passed')

## 5 · Original V7 YOLO proposal generation and cache compatibility

This is the actual V7 implementation. The only change is **`device=DEVICE`**, which enables GPU 0 when available. The original V7 cache schema, `max_det=65`, tile masks, and 12px truncation flags are retained.

In [ ]:
# If previous V13 output is reattached, it serves as the V7-compatible cache root.
PREVIOUS_V7_ROOT=PREVIOUS_V7_ROOT or (PREVIOUS_V13_ROOT if PREVIOUS_V13_ROOT else None)
PREVIOUS_V7_ROOT=str(PREVIOUS_V7_ROOT) if PREVIOUS_V7_ROOT else None
# Historical source output ZIP can be used for cached 30-screen images.
if USE_ARCHIVED_V7_ZIP and PREVIOUS_V7_ROOT is None:
    zip_candidates=list(Path('/kaggle/input').rglob('results - 2026-10-21T115631.189.zip'))
    if len(zip_candidates)==1:
        z=zipfile.ZipFile(zip_candidates[0]);prefix='solar_cpu_v7/'
        dest=OUT/'archived_v7';dest.mkdir(exist_ok=True)
        for name in z.namelist():
            if not name.startswith(prefix):continue
            rel=Path(name[len(prefix):])
            if not rel.parts or rel.parts[0] not in ('global','tiles') and str(rel)!='v7_config.json':continue
            if any(x in ('..','') for x in rel.parts):continue
            path=dest/rel;path.parent.mkdir(parents=True,exist_ok=True)
            if not path.exists() and not name.endswith('/'):path.write_bytes(z.read(name))
        if (dest/'v7_config.json').exists():PREVIOUS_V7_ROOT=str(dest)
        print('Attached historical V7 cache:',PREVIOUS_V7_ROOT)

In [ ]:
CONFIG={'checkpoint_sha256':MODEL_SHA,'split_sha256':SPLIT_SHA,'global_imgsz':GLOBAL_IMGSZ,
        'raw_conf':RAW_CONF,'raw_iou':RAW_IOU,'tile_imgsz':TILE_IMGSZ,'tile_side':TILE_SIDE,
        'tile_scheme':TILES,'model':'YOLO11n-seg','version':'v7-native-tiles-1'}
(OUT/'v7_config.json').write_text(json.dumps(CONFIG,indent=2))
PREV=Path(PREVIOUS_V7_ROOT) if PREVIOUS_V7_ROOT else None
if PREV:
    assert (PREV/'v7_config.json').is_file(), 'Attach previous V7 run output folder'
    assert json.loads((PREV/'v7_config.json').read_text())==CONFIG,'Previous V7 cache belongs to another config or checkpoint'

G_DIR=OUT/'global';T_DIR=OUT/'tiles';G_DIR.mkdir(exist_ok=True);T_DIR.mkdir(exist_ok=True)
MODEL=None

def get_model():
    global MODEL
    if MODEL is None:
        print('Loading historical YOLO11n-seg on:',CKPT)
        MODEL=YOLO(str(CKPT))
    return MODEL

def base_meta(kind,stem):
    return {**CONFIG,'stem':stem,'source':'LIVE_TEST_OR_TRAIN_JPEG_INFERENCE','kind':kind}

def get_cache(kind,stem):
    folder='global' if kind=='global' else 'tiles'
    paths=[OUT/folder/(stem+'.json')]
    if PREV:paths.append(PREV/folder/(stem+'.json'))
    if kind=='global':paths.append(SOURCE_V4_ROOT/'cache'/'yolo11n-seg'/(stem+'.json'))
    for p in paths:
        if not p.is_file():continue
        try:
            rec=json.loads(p.read_text())
            if p==paths[-1] and kind=='global' and p.parent.parent==SOURCE_V4_ROOT/'cache':
                # Original V4 cache schema: original checkpoint, global prediction size, TTA off.
                if (rec.get('stem')==stem and rec.get('checkpoint_sha256')==MODEL_SHA
                    and rec.get('predict_imgsz')==GLOBAL_IMGSZ and rec.get('tta') is False
                    and rec.get('source')=='LIVE_IMAGE_INFERENCE'):
                    return rec['proposals']
            else:
                if all(rec.get(k)==v for k,v in base_meta(kind,stem).items()):return rec['proposals']
        except Exception:continue
    return None

@torch.inference_mode()
def infer_view(gray,kind,x=0,y=0):
    assert gray.ndim==2
    inp=cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR)
    shape=gray.shape
    imgsz=GLOBAL_IMGSZ if kind=='global' else TILE_IMGSZ
    result=get_model().predict(source=inp,imgsz=imgsz,conf=RAW_CONF,iou=RAW_IOU,
        retina_masks=True,agnostic_nms=True,max_det=MAX_DET,device=DEVICE,verbose=False)[0]
    if result.boxes is None or result.masks is None:return []
    masks=result.masks.data.detach().cpu().numpy()
    scores=result.boxes.conf.detach().cpu().numpy()
    assert len(scores)==len(masks)
    result_list=[]
    for score,mask in zip(scores,masks):
        local=(mask>.5).astype(np.uint8)
        if local.shape!=shape:local=cv2.resize(local,(shape[1],shape[0]),interpolation=cv2.INTER_NEAREST)
        if not local.any():continue
        truncated=bool(has_internal_tile_edge(local,x,y)) if kind=='tile' else False
        full=local if kind=='global' else tile_mask_to_full(local,x,y)
        rr=encode(full)
        result_list.append({'counts':to_ascii(rr),'score':float(score),'area':int(coco.area(rr)),
                            'origin':kind,'x':int(x),'y':int(y),'truncated':truncated})
    return result_list

def infer_cached(kind,stem,image_path):
    already=get_cache(kind,stem)
    if already is not None:return already
    gray=cv2.imread(str(image_path),cv2.IMREAD_GRAYSCALE)
    if gray is None or gray.shape!=(H,W):raise RuntimeError('Image missing/shape mismatch: '+str(image_path))
    if kind=='global':found=infer_view(gray,'global')
    else:
        found=[]
        for x,y in TILE_WINDOWS:
            for p in infer_view(gray[y:y+TILE_SIDE,x:x+TILE_SIDE],'tile',x,y):found.append(p)
    rec={**base_meta(kind,stem),'proposals':found}
    dest=(G_DIR if kind=='global' else T_DIR)/(stem+'.json')
    dest.write_text(json.dumps(rec,separators=(',',':')))
    return found

def ensure(kind,stems,lookup):
    pending=[s for s in stems if get_cache(kind,s) is None]
    print(kind,'present:',len(stems)-len(pending),'pending:',len(pending))
    for stem in tqdm(pending,desc='CPU native-image '+kind+' predictions'):
        infer_cached(kind,stem,lookup[stem])

def pool(stem,mode):
    a=get_cache('global',stem)
    if a is None:raise FileNotFoundError('Global predictions absent: '+stem)
    a=[dict(p,origin='global',truncated=False) for p in a]
    if mode=='global':return a
    b=get_cache('tile',stem)
    if b is None:raise FileNotFoundError('Tile predictions absent: '+stem)
    return a+[dict(p,origin='tile',truncated=bool(p.get('truncated',False))) for p in b]

## 6 · Original V7 mask-selection policy, including containment/free-pixel tests

Copied directly from V7. Importantly, these checks do **not** match V12.1's simplified IoU suppression, so V12.1 could never be expected to reproduce the public CSV.

In [ ]:
def original_v4(stem):
    original=get_cache('global',stem)
    if original is None:raise FileNotFoundError(stem)
    entries=sorted([p for p in original if float(p['score'])>=.32 and int(p['area'])>=350],
                   key=lambda p:(-float(p['score']),-int(p['area'])))
    occupied=np.zeros((H,W),np.uint8);prior=[];out=[]
    for p in entries:
        r=to_rle(p['counts'])
        if prior:
            vv=np.asarray(coco.iou([r],prior,[0]*len(prior)))[0]
            if vv.size and vv.max()>=.50:continue
        prior.append(r)
        mm=coco.decode(r).astype(bool)&(occupied==0)
        if mm.sum()<350:continue
        occupied[mm]=1
        out.append(encode(mm))
    return out

def distinct(rles):
    if len(rles)<2:return True
    inter=np.asarray(coco.iou(rles,rles,[0]*len(rles)))
    np.fill_diagonal(inter,0)
    return bool(np.all(inter==0))

def decode_candidate(stem,p):
    vals=pool(stem,p['mode'])
    bonus=p['tile_bonus'];conf=p['global_conf'];tile_conf=p['tile_conf']
    kept=[]
    for q in vals:
        is_tile=q.get('origin')=='tile'
        if q['score']<(tile_conf if is_tile else conf):continue
        if q['area']<p['min_area']:continue
        if is_tile and p['seam_guard'] and q.get('truncated',False):continue
        kept.append({**q,'rank':float(q['score'])+(bonus if is_tile else 0.)})
    kept.sort(key=lambda q:(-q['rank'],-q['area']))
    occupied=np.zeros((H,W),np.uint8)
    previous=[];out=[]
    for q in kept:
        rr=to_rle(q['counts'])
        # Duplicate/containment check before expensive full-resolution mask decoding
        if previous:
            iou=np.asarray(coco.iou([rr],previous,[0]*len(previous)))[0]
            areas=np.asarray(coco.area(previous),dtype=np.float64)
            intersections=iou*(areas+float(q['area']))/(1+iou+1e-9)
            containment=intersections/max(1,float(q['area']))
            if np.any(iou>=p['nms_iou']) or np.any(containment>=p['containment']):continue
        mask=coco.decode(rr).astype(bool)
        free=mask&(occupied==0)
        count=int(free.sum())
        if count<p['min_area'] or count/max(1,float(q['area']))<p['min_free_fraction']:continue
        # Crucial: update the occupied map only for a successfully kept instance.
        occupied[free]=1
        enc=encode(free)
        previous.append(rr)
        out.append(enc)
    assert distinct(out)
    return out

# Synthetic duplicate/containment tests
m1=np.zeros((H,W),np.uint8);m1[20:100,20:150]=1
m2=np.zeros((H,W),np.uint8);m2[30:70,40:80]=1
r1,r2=encode(m1),encode(m2)
assert coco.iou([r1],[r2],[0])[0,0]>0
assert distinct([r1])
assert not distinct([r1,r2])
print('Original V4 and native-tile postprocessor initialized')

## 7 · Annotation pooling and V7-style local PQ scorer

All annotator versions of each physical observation are scored, then IoU/TP/FP/FN counts are pooled before dividing. **The organizer's latest self-evaluation routine remains authoritative** and parity must be verified before promotion.

In [ ]:
def annotation_rle(a):
    seg=a.get('segmentation')
    if isinstance(seg,dict):
        if isinstance(seg['counts'],list):return coco.frPyObjects(seg,H,W)
        return to_rle(seg['counts'])
    if not seg:return None
    if isinstance(seg[0],(int,float)):seg=[seg]
    polys=[]
    for poly in seg:
        if len(poly)<6 or len(poly)%2:continue
        pts=np.array(poly,dtype=np.float64).reshape(-1,2)
        if np.isfinite(pts).all():
            polys.append(np.clip(pts,[0,0],[W-1,H-1]).ravel().tolist())
    if not polys:return None
    ans=coco.frPyObjects(polys,H,W)
    return coco.merge(ans) if isinstance(ans,list) else ans

TRUTH_MEM={}
def truth(stem):
    if stem not in TRUTH_MEM:
        TRUTH_MEM[stem]=[]
        for annotator,items in OBS[stem]:
            TRUTH_MEM[stem].append((annotator,[r for item in items if (r:=annotation_rle(item)) is not None]))
    return TRUTH_MEM[stem]

def match_counts(gt,pred):
    if not gt and not pred:return np.zeros(4,dtype=np.float64)
    if not gt:return np.array([0,0,len(pred),0],dtype=np.float64)
    if not pred:return np.array([0,0,0,len(gt)],dtype=np.float64)
    ov=np.asarray(coco.iou(gt,pred,[0]*len(pred)),dtype=np.float64)
    matched=ov>0.5
    return np.array([ov[matched].sum(),matched.sum(),(matched.sum(axis=0)==0).sum(),
                     (matched.sum(axis=1)==0).sum()],dtype=np.float64)

def summarize(v):
    sum_iou,tp,fp,fn=map(float,v)
    denom=tp+.5*fp+.5*fn
    return {'pq':float(sum_iou/denom) if denom else 0.,'sq':float(sum_iou/tp) if tp else 0.,
            'rq':float(tp/denom) if denom else 0.,'tp':int(tp),'fp':int(fp),'fn':int(fn)}

def score(params,stems,keep_details=False):
    total=np.zeros(4,dtype=np.float64);rows=[]
    for stem in stems:
        pred=original_v4(stem) if params=='v4' else decode_candidate(stem,params)
        for version,gt in truth(stem):
            v=match_counts(gt,pred);total+=v
            if keep_details:rows.append({'stem':stem,'version':version,**summarize(v)})
    return summarize(total),pd.DataFrame(rows)

r1=encode(np.pad(np.ones((12,13),np.uint8),((10,H-22),(20,W-33))))
assert np.isclose(summarize(match_counts([r1],[r1]))['pq'],1.)
assert summarize(match_counts([r1],[]))['fn']==1
print('Multi-annotator proxy PQ unit tests passed')

## 8 · Recover the *historical* finalist family (not an invented search)

V7's original parameter grid is copied and checked against the archived screening index ordering: **38,39,36,37,34,35,14,15**, plus global-only and original V4 control. The original notebook was not the final CONFIRM/SUBMIT run, so we compare these specific recipes to the frozen 0.34 CSV on unlabeled test images.

In [ ]:
def parameter_grid():
    for gc,tc,bonus,guard,nms in product([.28,.36],[.35,.48,.62],[0.,.10],[False,True],[.50,.70]):
        yield {'mode':'tiles','global_conf':gc,'tile_conf':tc,'tile_bonus':bonus,
               'min_area':350,'seam_guard':guard,'nms_iou':nms,
               'containment':.88,'min_free_fraction':.50}
    yield {'mode':'global','global_conf':.32,'tile_conf':1.,'tile_bonus':0.,
           'min_area':350,'seam_guard':False,'nms_iou':.50,
           'containment':.88,'min_free_fraction':.50}
PARAMS=list(parameter_grid())
assert len(PARAMS)==49 and PARAMS[48]['mode']=='global'
SCREEN_FINALISTS=[38,39,36,37,34,35,14,15]
RECOVERY_RECIPES=[('v7_'+str(i),PARAMS[i]) for i in SCREEN_FINALISTS]
RECOVERY_RECIPES += [('v7_global',PARAMS[48]),('original_v4','v4')]
# The archived screening board confirms the exact order (not final submission identity).
HISTORICAL_V7_CONTRACT={'source_notebook_sha': '33b3ff8d118be547096f6146973eb8e618d1eff6e0f60cefe5e5ff4db01dcffc',
  'ultralytics':'8.4.175','checkpoint_sha256':MODEL_SHA,'split_sha256':SPLIT_SHA,
  'global_imgsz':1024,'tile_imgsz':1024,'tile_side':1280,'scheme':'FOUR',
  'raw_conf':.10,'raw_iou':.82,'max_det':65,'nearest_neighbor_mask_resize':True,
  'internal_tile_margin':12,'min_instance_area':350,
  'screen_top_indices':SCREEN_FINALISTS,'candidate_search':RECOVERY_RECIPES,
  'last_selection_json_supplied':False,'test_labels_used':False}
(OUT/'historical_v7_contract.json').write_text(json.dumps(HISTORICAL_V7_CONTRACT,indent=2,default=str))
print('Historical recipes:',[name for name,_ in RECOVERY_RECIPES])

## 9 · Reproduction agreement calculation

The score uses per-image Hungarian IoU to compare our predicted RLE masks against *your own frozen masks*. It requires **identical counts on all 180 images and mean matched IoU ≥ .995**, preserving the original CSV as the only truth of past inference. No test ground-truth annotations are consulted.

In [ ]:
def rles_frozen(stem):return [to_rle(c) for c in FROZEN_RLES.get(stem,[])]

def reproduction_score(reference,candidate):
    nr,nc=len(reference),len(candidate)
    if not nr and not nc:return 0.,0,True,1.
    if not nr or not nc:return 0.,max(nr,nc),False,0.
    mat=np.asarray(coco.iou(reference,candidate,[0]*nc),np.float64)
    ri,ci=linear_sum_assignment(-mat)
    total_iou=float(mat[ri,ci].sum())
    return total_iou,max(nr,nc),nr==nc,total_iou/max(nr,nc)

def evaluate_recovery(stems,recipe,name,per_image=False):
    total_iou=0.;denom=0.;same=0;records=[]
    for stem in stems:
        candidate=original_v4(stem) if recipe=='v4' else decode_candidate(stem,recipe)
        ref=rles_frozen(stem)
        s,n,match,agr=reproduction_score(ref,candidate)
        total_iou+=s;denom+=n;same+=match
        if per_image:
            records.append({'stem':stem,'recipe':name,'expected_count':len(ref),
                            'recovered_count':len(candidate),'same_count':bool(match),
                            'matched_iou':s,'mask_slots':n,'image_agreement':agr})
    return {'recipe':name,'agreement':total_iou/max(1,denom),'same_counts':same,
            'images':len(stems),'reference_rows':sum(len(rles_frozen(s)) for s in stems)}, records

## 10 · Run test-image recovery; save full forensic data even on failure

This stage can use GPU for YOLO inference. **No action model trains** in RECOVER mode. Cache is deterministic and keyed to the original V7 configuration and model SHA.

In [ ]:
RECOVERY_PATH=OUT/'recovery'/'baseline_recovery.json'
RECOVERY=None

def run_recovery():
    ensure('global',sorted(TEST),TEST)
    ensure('tile',sorted(TEST),TEST)
    summary=[];all_details=[]
    for name,recipe in tqdm(RECOVERY_RECIPES,desc='Exact historical finalist recovery'):
        result,details=evaluate_recovery(sorted(TEST),recipe,name,per_image=True)
        summary.append(result);all_details.extend(details)
        print(name,'agreement',round(result['agreement'],6),
              'matching count images',result['same_counts'],'/180')
    table=pd.DataFrame(summary).sort_values(['same_counts','agreement'],ascending=False)
    table.to_csv(OUT/'recovery'/'baseline_recovery_results.csv',index=False)
    pd.DataFrame(all_details).to_csv(OUT/'recovery'/'best_recipe_per_image.csv',index=False)
    winner=table.iloc[0].to_dict()
    selected=next(p for name,p in RECOVERY_RECIPES if name==winner['recipe'])
    recovered=bool(winner['same_counts']==180 and winner['agreement']>=.995)
    result={'recovered':recovered,'recipe_name':winner['recipe'],'recipe':selected,
            'agreement':float(winner['agreement']),'same_counts':int(winner['same_counts']),
            'total_images':180,'baseline_sha256':BASE_SHA,'checkpoint_sha256':MODEL_SHA,
            'split_sha256':SPLIT_SHA,'config':CONFIG,'ultralytics':ultralytics.__version__,
            'candidate_family':'ACTUAL_V7_SCREENING_FINALISTS_PLUS_CONTROLS',
            'hidden_test_labels_used':False}
    RECOVERY_PATH.write_text(json.dumps(result,indent=2,default=str))
    print('BEST BASELINE RECOVERY:',json.dumps(result,indent=2,default=str))
    worst=pd.DataFrame(all_details)
    worst=worst[worst.recipe==winner['recipe']].sort_values('image_agreement').head(30)
    worst.to_csv(OUT/'diagnostics'/'recovery_worst_images.csv',index=False)
    (OUT/'diagnostics'/'recovery_forensics_summary.json').write_text(json.dumps({
        'candidate_count':len(RECOVERY_RECIPES),
        'best_recovered':recovered,'best_recipe':winner['recipe'],
        'count_mismatched_images':180-int(winner['same_counts']),
        'next_steps':'Compare original V7 selection.json or raw masks when recovery fails; NEVER promote while failing.'
    },indent=2))
    return result

if MODE in ('RECOVER','ALL'):
    RECOVERY=run_recovery()
else:
    if RECOVERY_PATH.is_file():RECOVERY=json.loads(RECOVERY_PATH.read_text())
    elif PREVIOUS_V13_ROOT and (PREVIOUS_V13_ROOT/'recovery'/'baseline_recovery.json').exists():
        RECOVERY=json.loads((PREVIOUS_V13_ROOT/'recovery'/'baseline_recovery.json').read_text())
    else:
        raise FileNotFoundError('Run MODE="RECOVER" first; attach its saved output to use ACTIONS/SUBMIT.')
assert RECOVERY['baseline_sha256']==BASE_SHA and RECOVERY['checkpoint_sha256']==MODEL_SHA
assert RECOVERY['split_sha256']==SPLIT_SHA and RECOVERY['config']==CONFIG
RECIPE=RECOVERY['recipe']
print('Baseline recovered:',RECOVERY['recovered'])

## 11 · Archived-V7 versus live proposals diagnostics (optional)

If a V7 ZIP is attached, compare live vs cached raw proposals. This catches package-dependent inference drift separately from differences in mask selection.

In [ ]:
def raw_cache_parity(stems):
    rows=[]
    if not PREVIOUS_V7_ROOT:return rows
    for stem in stems:
        for kind in ('global','tiles'):
            old_file=Path(PREVIOUS_V7_ROOT)/kind/(stem+'.json')
            new_file=(G_DIR if kind=='global' else T_DIR)/(stem+'.json')
            if not (old_file.exists() and new_file.exists()):continue
            old_rec=json.loads(old_file.read_text());new_rec=json.loads(new_file.read_text())
            old_p=old_rec['proposals'];new_p=new_rec['proposals']
            ov=[to_rle(q['counts']) for q in old_p]
            nv=[to_rle(q['counts']) for q in new_p]
            s,n,m,a=reproduction_score(ov,nv)
            rows.append({'stem':stem,'kind':kind,'old':len(ov),'new':len(nv),
                         'identical_counts':m,'agreement':a})
    if rows:pd.DataFrame(rows).to_csv(OUT/'diagnostics'/'historical_raw_proposal_parity.csv',index=False)
    return rows
if MODE in ('RECOVER','ALL'):
    _parity=raw_cache_parity(sorted(TEST)[:16])
    print('Historical raw-proposal parity available for',len(_parity),'cached comparisons')

## 12 · A second, strict one-to-one Panoptic Quality scorer

V7's local scorer counts IoU>0.5 matches. Because the official evaluator can differ, action evaluation below uses a **one-to-one Hungarian matching** and pools TP/FP/FN across annotators. **Promotion still requires a literal organizer scorer comparison** (`OFFICIAL_PQ_FUNCTION` plus `CONFIRMED_OFFICIAL_PQ_PARITY=True`).

In [ ]:
def pq_counts(gt,pred):
    if not gt and not pred:return np.zeros(4,np.float64)
    if not gt:return np.array([0.,0.,float(len(pred)),0.])
    if not pred:return np.array([0.,0.,0.,float(len(gt))])
    ov=np.asarray(coco.iou(gt,pred,[0]*len(pred)),np.float64)
    ri,ci=linear_sum_assignment(-ov)
    mask=ov[ri,ci]>.5
    tp=int(mask.sum())
    return np.array([float(ov[ri[mask],ci[mask]].sum()),float(tp),
                     float(len(pred)-tp),float(len(gt)-tp)])

def pooled_counts(stem,pred):
    out=np.zeros(4,np.float64)
    for _,version in truth(stem):out+=pq_counts(version,pred)
    return out

def pq_score(c):
    s,tp,fp,fn=map(float,c)
    denom=tp+.5*(fp+fn)
    return s/denom if denom>0 else 0.

def pq_summary(c):
    s,tp,fp,fn=map(float,c)
    den=tp+.5*(fp+fn)
    return {'pq':pq_score(c),'sq':s/tp if tp else 0.,
            'rq':tp/den if den else 0.,'tp':int(tp),'fp':int(fp),'fn':int(fn)}

# Synthetic scorer sanity tests and one-to-one matching.
synthetic=np.zeros((H,W),np.uint8);synthetic[3:27,5:53]=1
r=encode(synthetic)
assert np.allclose(pq_counts([r],[r]),[1,1,0,0])
assert np.allclose(pq_counts([r],[]),[0,0,0,1])
assert np.allclose(pq_counts([],[r]),[0,0,1,0])
assert float(coco.area(r))==int(synthetic.sum())
# Optional literal reference adapter. Paste the organizer's official per-image PQ
# function below; it should accept (gt_rles, predicted_rles) and return PQ.
# Do not assume a self-written Hungarian scorer is identical to the organizer.
OFFICIAL_PQ_FUNCTION = None

def official_evaluator_parity(reference_fn):
    if reference_fn is None:return False
    m2=np.zeros((H,W),np.uint8);m2[120:160,90:140]=1
    second=encode(m2)
    cases=[([r],[r]),([r],[]),([],[r]),([r],[second]),
           ([r,second],[r,second]),([r,second],[r]),([],[])]
    for gt,pr in cases:
        expected=pq_score(pq_counts(gt,pr))
        observed=float(reference_fn(gt,pr))
        if not np.isclose(expected,observed,rtol=0,atol=1e-9):
            raise AssertionError(f'Organizer evaluator mismatch: local={expected}, official={observed}')
    return True

OFFICIAL_PARITY_OK=official_evaluator_parity(OFFICIAL_PQ_FUNCTION)
print('PQ/RLE synthetic tests passed. Official parity:',OFFICIAL_PARITY_OK)

## 13 · Candidate bank (unused V7 proposals + conservative multi-scale dark-ridge features)

No ridge component is ever directly submitted. Each ADD candidate must pass the trained action gate. Optional 896-pixel microtiles are feature/proposal generators, **not** replacements for the baseline.

In [ ]:
CLAHE=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))

def ridge_response(gray):
    normalized=CLAHE.apply(gray)
    outputs=[]
    for k in (9,17,33):
        kernel=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(k,k))
        outputs.append(cv2.morphologyEx(normalized,cv2.MORPH_BLACKHAT,kernel))
    return np.maximum.reduce(outputs)

def ridge_candidates(ridge):
    if not ENABLE_RIDGE_ADD:return []
    cut=max(12.,float(np.percentile(ridge,99.2)))
    binary=(ridge>=cut).astype(np.uint8)
    binary=cv2.morphologyEx(binary,cv2.MORPH_CLOSE,
                            cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5)))
    n,labels,stats,centroids=cv2.connectedComponentsWithStats(binary,8)
    entries=[]
    for j in range(1,n):
        a=int(stats[j,cv2.CC_STAT_AREA])
        if a<220 or a>90000:continue
        x,y,w,h=map(int,stats[j,:4]);cx,cy=centroids[j]
        if np.hypot(cx-1024,cy-1024)>990:continue
        mask=np.zeros((H,W),np.uint8)
        mask[y:y+h,x:x+w]=(labels[y:y+h,x:x+w]==j).astype(np.uint8)
        score=float(ridge[y:y+h,x:x+w][labels[y:y+h,x:x+w]==j].mean()/255.)
        e=encode(mask)
        entries.append({'counts':to_ascii(e),'score':score,'area':a,'origin':'ridge',
                        'x':0,'y':0,'truncated':False})
    entries.sort(key=lambda q:q['score']*np.log1p(q['area']),reverse=True)
    return entries[:18]

MICROTILE_SIDE=896

def microtile_proposals(stem,image_path,base,ridge):
    if not ENABLE_MICROTILES:return []
    # Only 3 deterministic high-ridge spots outside existing baseline masks.
    gray=cv2.imread(str(image_path),cv2.IMREAD_GRAYSCALE)
    occupied=np.zeros((H,W),np.uint8)
    for r in base:occupied|=coco.decode(r).astype(np.uint8)
    score=ridge.copy();score[occupied>0]=0
    spots=[]
    for _ in range(3):
        i=int(np.argmax(score));yy,xx=divmod(i,W)
        if score[yy,xx]<15:break
        x=int(np.clip(xx-MICROTILE_SIDE//2,0,W-MICROTILE_SIDE))
        y=int(np.clip(yy-MICROTILE_SIDE//2,0,H-MICROTILE_SIDE))
        spots.append((x,y))
        cv2.circle(score,(xx,yy),160,0,-1)
    result=[]
    for x,y in sorted(set(spots)):
        crop=gray[y:y+MICROTILE_SIDE,x:x+MICROTILE_SIDE]
        ans=get_model().predict(source=cv2.cvtColor(crop,cv2.COLOR_GRAY2BGR),imgsz=1024,
             conf=RAW_CONF,iou=RAW_IOU,max_det=MAX_DET,retina_masks=True,
             agnostic_nms=True,device=DEVICE,verbose=False)[0]
        if ans.masks is None:continue
        for conf,pm in zip(ans.boxes.conf.detach().cpu().numpy(),ans.masks.data.detach().cpu().numpy()):
            local=(pm>.5).astype(np.uint8)
            if local.shape!=crop.shape:
                local=cv2.resize(local,crop.shape[::-1],interpolation=cv2.INTER_NEAREST)
            if int(local.sum())<350:continue
            full=np.zeros((H,W),np.uint8)
            full[y:y+MICROTILE_SIDE,x:x+MICROTILE_SIDE]=local
            rr=encode(full)
            result.append({'counts':to_ascii(rr),'score':float(conf),'area':int(coco.area(rr)),
                           'origin':'micro','x':x,'y':y,'truncated':False})
    return result

## 14 · Reconstruct baseline with original V7 rules and attach proposal metadata

Selected masks are exactly those output by `decode_candidate`. The action model attaches the nearest supporting source/score to each mask **without changing its pixels**. A regression assertion detects any discrepancy.

In [ ]:
def baseline_with_meta(stem):
    rles=original_v4(stem) if RECIPE=='v4' else decode_candidate(stem,RECIPE)
    raw=pool(stem,'tiles')
    all_r=[to_rle(q['counts']) for q in raw]
    matched=[]
    if rles and all_r:
        mat=np.asarray(coco.iou(rles,all_r,[0]*len(all_r)))
        for r,row in zip(rles,mat):
            j=int(row.argmax());p=dict(raw[j]);p['counts']=to_ascii(r)
            p['area']=int(coco.area(r));matched.append(p)
    else:
        matched=[{'counts':to_ascii(r),'area':int(coco.area(r)),'score':0.,
                  'origin':'unknown','x':0,'y':0,'truncated':False} for r in rles]
    assert len(matched)==len(rles)
    assert all(m['counts']==to_ascii(r) for m,r in zip(matched,rles))
    return matched

def add_proposals(stem,gray,ridge,base,raw):
    occupied=np.zeros((H,W),np.uint8)
    for q in base:occupied|=coco.decode(to_rle(q['counts'])).astype(np.uint8)
    candidates=sorted(list(raw)+ridge_candidates(ridge)+microtile_proposals(
        stem,TRAIN.get(stem,TEST.get(stem)),[to_rle(q['counts']) for q in base],ridge),
        key=lambda p:p['score'],reverse=True)
    accepted=[];base_r=[to_rle(q['counts']) for q in base]
    for p in candidates:
        r=to_rle(p['counts'])
        if base_r:
            ov=np.asarray(coco.iou([r],base_r,[0]*len(base_r)))[0]
            if ov.size and float(ov.max())>.28:continue
        mask=coco.decode(r).astype(bool)&(occupied==0)
        if int(mask.sum())<240:continue
        e=encode(mask)
        if accepted:
            ov=np.asarray(coco.iou([e],[to_rle(q['counts']) for q in accepted],
                                      [0]*len(accepted)))[0]
            if ov.size and float(ov.max())>.55:continue
        accepted.append({**p,'counts':to_ascii(e),'area':int(coco.area(e))})
        if len(accepted)>=28:break
    return accepted

def pair_candidates(base,max_gap=85):
    boxes=[coco.toBbox(to_rle(q['counts'])).astype(float) for q in base]
    pairs=[]
    for i in range(len(base)):
        for j in range(i+1,len(base)):
            x,y,w,h=boxes[i];a,b,c,d=boxes[j]
            dx=max(0.,x-(a+c),a-(x+w))
            dy=max(0.,y-(b+d),b-(y+h))
            gap=float(np.hypot(dx,dy))
            if gap<=max_gap:pairs.append((gap,i,j))
    pairs.sort()
    return [(i,j) for _,i,j in pairs[:30]]

## 15 · Thin-structure appearance, topology and pair features

We use mask skeleton length, endpoints, width, local contrast, sun-disk radius, ridge energy, source consensus, and pair geometry/orientation, matching the report's feature recommendations. Feature column definitions are fixed across folds and inference.

In [ ]:
MASK_COLS=['score','global','tile','ridge','micro','truncated','log_area','log_skel',
 'endpoints','branches','width','aspect','fill','contrast','ridge_mean',
 'radial_distance','support_iou','overlap_baseline']
PAIR_COLS=['endpoint_gap','bbox_gap','center_gap','area_ratio','axis_parallel',
 'join_a','join_b','mean_gray','mean_ridge','radial_delta']
ACTION_SIGNATURE=hashlib.sha256(json.dumps({'feature_cols':MASK_COLS,'pair_cols':PAIR_COLS,
  'microtiles':ENABLE_MICROTILES,'ridge_add':ENABLE_RIDGE_ADD,'model_sha256':MODEL_SHA,
  'raw_config':CONFIG,'ultralytics':ultralytics.__version__},sort_keys=True).encode()).hexdigest()

def geometry(rle):
    m=coco.decode(to_rle(rle)).astype(bool)
    ys,xs=np.where(m)
    if len(xs)==0:return None
    x0,x1=int(xs.min()),int(xs.max())+1
    y0,y1=int(ys.min()),int(ys.max())+1
    crop=m[y0:y1,x0:x1]
    sk=skeletonize(crop)
    neighbors=(cv2.filter2D(sk.astype(np.uint8),cv2.CV_16S,np.ones((3,3),np.uint8))
               -sk.astype(np.int16))
    ey,ex=np.where(sk&(neighbors==1))
    endpoint=np.column_stack((ex+x0,ey+y0)).astype(np.float32) if len(ex) else np.empty((0,2),np.float32)
    sy,sx=np.where(sk)
    if len(sx)>2:
        cloud=np.column_stack((sx,sy)).astype(np.float32)
        vals,vecs=np.linalg.eigh(np.cov(cloud.T))
        axis=vecs[:,int(np.argmax(vals))].astype(np.float32)
    else:axis=np.asarray([1.,0.],np.float32)
    return {'area':len(xs),'bbox':(x0,y0,x1,y1),'mask':m,
      'skel_len':int(sk.sum()),'endpoint':endpoint,'branches':int((sk&(neighbors>=3)).sum()),
      'axis':axis,'center':np.asarray([float(xs.mean()),float(ys.mean())],np.float32)}

def mask_features(gray,ridge,p,raw,base,action):
    g=geometry(p['counts'])
    if g is None:return np.zeros(len(MASK_COLS),np.float32)
    x0,y0,x1,y1=g['bbox'];m=g['mask'];roi=m[max(0,y0-6):min(H,y1+6),max(0,x0-6):min(W,x1+6)]
    yy=max(0,y0-6);xx=max(0,x0-6)
    ring=cv2.dilate(roi.astype(np.uint8),np.ones((11,11),np.uint8)).astype(bool)&~roi
    local=gray[yy:yy+roi.shape[0],xx:xx+roi.shape[1]]
    inside=float(gray[m].mean())
    bg=float(local[ring].mean()) if ring.any() else inside
    contrast=(bg-inside)/max(12.,float(local.std()))
    others=[to_rle(q['counts']) for q in raw if q['origin']!=p.get('origin')]
    own=to_rle(p['counts'])
    support=float(np.max(coco.iou([own],others,[0]*len(others)))) if others else 0.
    bs=[to_rle(q['counts']) for q in base]
    overlap=float(np.max(coco.iou([own],bs,[0]*len(bs)))) if bs else 0.
    bw=x1-x0;bh=y1-y0
    vals=[float(p.get('score',0.)),float(p.get('origin')=='global'),
          float(p.get('origin')=='tile'),float(p.get('origin')=='ridge'),
          float(p.get('origin')=='micro'),float(p.get('truncated',False)),
          np.log1p(g['area']),np.log1p(g['skel_len']),len(g['endpoint']),g['branches'],
          g['area']/max(1,g['skel_len']),max(bw,bh)/max(1,min(bw,bh)),
          g['area']/max(1,bw*bh),contrast,float(ridge[m].mean()/255.),
          float(np.linalg.norm(g['center']-[1024,1024])/1024.),support,overlap]
    return np.nan_to_num(np.asarray(vals,np.float32))

def pair_features(gray,ridge,a,b):
    ga=geometry(a['counts']);gb=geometry(b['counts'])
    if ga is None or gb is None:return np.zeros(len(PAIR_COLS),np.float32)
    ea=ga['endpoint'] if len(ga['endpoint']) else ga['center'][None,:]
    eb=gb['endpoint'] if len(gb['endpoint']) else gb['center'][None,:]
    dist=np.linalg.norm(ea[:,None,:]-eb[None,:,:],axis=2)
    ia,ib=np.unravel_index(np.argmin(dist),dist.shape)
    pa,pb=ea[ia],eb[ib];v=pb-pa;gap=float(np.linalg.norm(v));uv=v/max(1e-6,gap)
    n=max(2,int(gap)+1)
    path=np.linspace(pa,pb,n)
    xx=np.clip(path[:,0].astype(int),0,W-1);yy=np.clip(path[:,1].astype(int),0,H-1)
    x0,y0,x1,y1=ga['bbox'];a0,b0,a1,b1=gb['bbox']
    bbox_gap=np.hypot(max(0,x0-a1,a0-x1),max(0,y0-b1,b0-y1))
    ra=float(np.linalg.norm(ga['center']-[1024,1024])/1024.)
    rb=float(np.linalg.norm(gb['center']-[1024,1024])/1024.)
    vals=[gap,float(bbox_gap),float(np.linalg.norm(ga['center']-gb['center'])),
          min(ga['area'],gb['area'])/max(ga['area'],gb['area']),
          abs(float(np.dot(ga['axis'],gb['axis']))),abs(float(np.dot(ga['axis'],uv))),
          abs(float(np.dot(gb['axis'],uv))),float(gray[yy,xx].mean()/255.),
          float(ridge[yy,xx].mean()/255.),abs(ra-rb)]
    return np.nan_to_num(np.asarray(vals,np.float32))

## 16 · Prepare one observation, with literal ΔPQ action targets

Labels are **change in pooled Panoptic Quality** when removing an existing mask, adding a new proposal, or merging two baseline masks. This trains the actual competition economics instead of relying on YOLO confidence alone.

In [ ]:
def feature_arrays(gray,ridge,raw,base,adds,pairs):
    return {
      'delete':np.asarray([mask_features(gray,ridge,p,raw,base,'delete') for p in base],
                          dtype=np.float32).reshape(-1,len(MASK_COLS)),
      'add':np.asarray([mask_features(gray,ridge,p,raw,base,'add') for p in adds],
                       dtype=np.float32).reshape(-1,len(MASK_COLS)),
      'merge':np.asarray([pair_features(gray,ridge,base[i],base[j]) for i,j in pairs],
                         dtype=np.float32).reshape(-1,len(PAIR_COLS))}

def prepare(stem,label=True):
    img_path=TRAIN[stem] if label else (TEST.get(stem) or TRAIN.get(stem))
    if img_path is None:raise KeyError(stem)
    gray=cv2.imread(str(img_path),cv2.IMREAD_GRAYSCALE)
    if gray is None or gray.shape!=(H,W):raise RuntimeError(img_path)
    ridge=ridge_response(gray)
    raw=pool(stem,'tiles')
    base=baseline_with_meta(stem)
    adds=add_proposals(stem,gray,ridge,base,raw)
    pairs=pair_candidates(base)
    features=feature_arrays(gray,ridge,raw,base,adds,pairs)
    obj={'base':base,'adds':adds,'pairs':pairs,'features':features}
    if label:
        rles=[to_rle(p['counts']) for p in base]
        old=pq_score(pooled_counts(stem,rles))
        deltas={'delete':[],'add':[],'merge':[]}
        for i in range(len(base)):
            others=[r for j,r in enumerate(rles) if i!=j]
            deltas['delete'].append(pq_score(pooled_counts(stem,others))-old)
        for p in adds:
            delta=pq_score(pooled_counts(stem,rles+[to_rle(p['counts'])]))-old
            deltas['add'].append(delta)
        for i,j in pairs:
            mi=coco.decode(rles[i]).astype(bool);mj=coco.decode(rles[j]).astype(bool)
            combined=encode(mi|mj)
            others=[r for k,r in enumerate(rles) if k not in (i,j)]
            deltas['merge'].append(pq_score(pooled_counts(stem,others+[combined]))-old)
        obj['labels']={k:np.asarray([float(v)>1e-10 for v in arr],np.uint8)
                       for k,arr in deltas.items()}
        obj['deltas']=deltas
    return obj

## 17 · Grouped acquisition-day two-fold cross-fitting and per-action HistGBM

Fold grouping never places the same acquisition-day prefix in train and validation. Empty ADD or MERGE feature arrays are explicitly supported (the V12.1 hotfix). Constant-label actions use a safe constant-probability fallback.

In [ ]:
class ConstantProbability:
    def __init__(self,p=0.):self.p=float(p)
    def predict_proba(self,X):
        n=len(X)
        return np.column_stack((np.full(n,1-self.p),np.full(n,self.p)))

def fold_groups(stems):
    groups=defaultdict(list)
    for stem in stems:groups[stem[:8]].append(stem)
    keys=sorted(groups)
    random.Random(SEED).shuffle(keys)
    folds=[[],[]]
    for k in sorted(keys,key=lambda v:len(groups[v]),reverse=True):
        folds[0 if len(folds[0])<=len(folds[1]) else 1]+=groups[k]
    assert folds[0] and folds[1]
    assert not ({x[:8] for x in folds[0]}&{x[:8] for x in folds[1]})
    return [sorted(x) for x in folds]

FOLDS=fold_groups(SPLIT['tune'])
assert len(FOLDS[0])+len(FOLDS[1])==97
print('Independent acquisition-day action folds:',[len(x) for x in FOLDS])

def fit_action_models(stems,observations):
    models={}
    for action in ('delete','add','merge'):
        blocks=[observations[s]['features'][action] for s in stems
                if len(observations[s]['features'][action])]
        labels=[observations[s]['labels'][action] for s in stems
                if len(observations[s]['labels'][action])]
        if not blocks:
            models[action]=ConstantProbability(0.);continue
        X=np.concatenate(blocks,axis=0);y=np.concatenate(labels,axis=0)
        if len(y)<30 or len(np.unique(y))<2:
            models[action]=ConstantProbability(y.mean() if len(y) else 0.);continue
        npos=max(1,int(y.sum()));nneg=int((y==0).sum())
        weight=np.where(y==1,min(10.,nneg/npos),1.)
        clf=HistGradientBoostingClassifier(learning_rate=.05,max_iter=160,
              max_leaf_nodes=15,max_depth=5,min_samples_leaf=18,
              l2_regularization=6.,random_state=SEED)
        clf.fit(X,y,sample_weight=weight)
        models[action]=clf
    return models

def action_probabilities(models,data):
    return {name:(models[name].predict_proba(data['features'][name])[:,1]
                  if len(data['features'][name]) else np.zeros(0,np.float64))
            for name in ('delete','add','merge')}

## 18 · Deterministic, non-overlapping DELETE → MERGE → ADD policy

Edits are conflict-free; no merge uses a deleted mask or already-merged mask. ADD masks are clipped against occupied pixels and must retain minimum area. `delta_ledger` precisely records applied actions.

In [ ]:
POLICIES=[{'delete':d,'add':a,'merge':m}
          for d in (.80,.90) for a in (.90,.95) for m in (.80,.90)]

def policy_predict(data,probs,policy):
    base=data['base'];adds=data['adds'];pairs=data['pairs']
    remove={i for i,p in enumerate(probs['delete']) if p>=policy['delete']}
    merged=[];used=set()
    for k in np.argsort(-probs['merge']):
        k=int(k)
        if probs['merge'][k]<policy['merge']:break
        i,j=pairs[k]
        if i in remove or j in remove or i in used or j in used:continue
        merged.append((i,j));used|={i,j}
    output=[to_rle(p['counts']) for k,p in enumerate(base) if k not in remove|used]
    for i,j in merged:
        m=coco.decode(to_rle(base[i]['counts'])).astype(bool)|coco.decode(to_rle(base[j]['counts'])).astype(bool)
        output.append(encode(m))
    occupied=np.zeros((H,W),np.uint8)
    for r in output:occupied|=coco.decode(r).astype(np.uint8)
    add_count=0
    for idx in np.argsort(-probs['add']):
        idx=int(idx)
        if probs['add'][idx]<policy['add']:break
        mask=coco.decode(to_rle(adds[idx]['counts'])).astype(bool)&(occupied==0)
        if int(mask.sum())<240:continue
        output.append(encode(mask));occupied[mask]=1;add_count+=1
    ledger={'delete':len(remove),'merge':len(merged),'add':add_count,
            'original':len(base),'final':len(output)}
    return output,ledger

def masks_disjoint(rles):
    # Exact pixels, not approximate bounding-box checks.
    occupied=np.zeros((H,W),bool)
    for r in rles:
        m=coco.decode(to_rle(r)).astype(bool)
        if not m.any() or np.any(m&occupied):return False
        occupied|=m
    return True

## 19 · Train action models and produce grouped OOF metrics

The local 97-observation evaluation is **not public leaderboard validation**. Eight joint policies are evaluated on the same out-of-fold predictions; any chosen policy remains subject to the historical-audit diagnostic and the exact recovery gate.

In [ ]:
ACTION_MODE=MODE in ('ACTIONS','ALL')
TUNE_DATA={};OOF={};FINAL_MODELS={}
ACTION_SELECTION_PATH=OUT/'selection.json'

def run_actions():
    stems=sorted(SPLIT['tune'])
    ensure('global',stems,TRAIN)
    ensure('tile',stems,TRAIN)
    observations={}
    for s in tqdm(stems,desc='Direct Delta-PQ action labeling (97 observations)'):
        observations[s]=prepare(s,label=True)
    for action in ('delete','add','merge'):
        rows=[{'stem':s,'index':i,'delta_pq':float(value),'positive':int(value>1e-10)}
              for s in stems for i,value in enumerate(observations[s]['deltas'][action])]
        pd.DataFrame(rows,columns=['stem','index','delta_pq','positive']).to_csv(
              OUT/'action_tables'/(action+'.csv'),index=False)
        print(action,'total',len(rows),'positive',sum(row['positive'] for row in rows))
    oof={}
    for fold in (0,1):
        learner=fit_action_models(FOLDS[1-fold],observations)
        for action,model in learner.items():
            joblib.dump(model,OUT/'models'/f'{action}_fold{fold}.joblib')
        for s in FOLDS[fold]:oof[s]=action_probabilities(learner,observations[s])
    baseline_counts={s:pooled_counts(s,[to_rle(p['counts']) for p in observations[s]['base']])
                     for s in stems}
    base_pooled=sum(baseline_counts.values(),np.zeros(4,np.float64))
    base_pq=pq_score(base_pooled)
    records=[]
    for idx,policy in enumerate(tqdm(POLICIES,desc='Eight OOF policies')):
        pooled=np.zeros(4,np.float64)
        by_fold=[np.zeros(4,np.float64) for _ in range(2)]
        for fold,fold_stems in enumerate(FOLDS):
            for s in fold_stems:
                rles,ledger=policy_predict(observations[s],oof[s],policy)
                score_vec=pooled_counts(s,rles)
                pooled+=score_vec;by_fold[fold]+=score_vec
        base_folds=[sum((baseline_counts[s] for s in ff),np.zeros(4,np.float64)) for ff in FOLDS]
        gains=[pq_score(by_fold[i])-pq_score(base_folds[i]) for i in range(2)]
        records.append({'index':idx,**policy,**pq_summary(pooled),
                        'gain':pq_score(pooled)-base_pq,'fold0_gain':gains[0],
                        'fold1_gain':gains[1]})
    table=pd.DataFrame(records).sort_values(['pq','rq'],ascending=False)
    table.to_csv(OUT/'tuning_results.csv',index=False)
    display(table)
    top=table.iloc[0]
    chosen=POLICIES[int(top['index'])]
    final_models=fit_action_models(stems,observations)
    for action,model in final_models.items():joblib.dump(model,OUT/'models'/f'{action}_final.joblib')
    print('Reconstructed baseline pooled OOF PQ:',base_pq)
    print('Candidate OOF PQ:',float(top['pq']),'absolute gain:',float(top['gain']))
    return {'baseline':base_pq,'policy':chosen,'candidate':float(top['pq']),
            'gain':float(top['gain']),'fold_gains':[float(top['fold0_gain']),
                                                     float(top['fold1_gain'])],
            'models':final_models,'observations':observations,'oof':oof}

ACTIONS=run_actions() if ACTION_MODE else None

## 20 · Feature/action ablations and calibration diagnostics

Ablation plots explain whether DELETE, ADD or MERGE causes the gain, and whether adding more proposals merely increases false positives. No test labels are used for analysis.

In [ ]:
def action_ablation_report(info):
    observations=info['observations'];oof=info['oof'];p=info['policy'];rows=[]
    for active in [('delete',),('add',),('merge',),('delete','add'),('delete','merge'),('add','merge'),('delete','add','merge')]:
        total=np.zeros(4,np.float64)
        for s,d in observations.items():
            probs={name:(oof[s][name] if name in active else np.zeros_like(oof[s][name]))
                   for name in ('delete','add','merge')}
            rles,_=policy_predict(d,probs,p)
            total+=pooled_counts(s,rles)
        rows.append({'actions':'+'.join(active),**pq_summary(total),
                     'gain':pq_score(total)-info['baseline']})
    pd.DataFrame(rows).sort_values('gain',ascending=False).to_csv(
        OUT/'diagnostics'/'oof_action_ablations.csv',index=False)
    source_counts=defaultdict(lambda:[0,0])
    for d in observations.values():
        for add,label in zip(d['adds'],d['labels']['add']):
            rec=source_counts[add['origin']];rec[0]+=1;rec[1]+=int(label)
    pd.DataFrame([{'source':k,'total':v[0],'positive':v[1],
                   'positive_rate':v[1]/max(1,v[0])}
                  for k,v in source_counts.items()]).to_csv(
        OUT/'diagnostics'/'add_source_summary.csv',index=False)
if ACTIONS:action_ablation_report(ACTIONS)

## 21 · Previously inspected 97-image audit (diagnostic, not pristine validation)

The audit is run **even if historical recipe recovery fails**, because the V12.1 results are worth checking independently. But promotion remains prohibited while recovery fails. Audit is not used for tuning thresholds or selecting models.

In [ ]:
AUDIT_RESULT={'ran':False,'baseline_pq':None,'candidate_pq':None,'delta':None,
              'note':'Historical audit has been previously inspected; diagnostic only.'}

def run_audit(info):
    stems=sorted(SPLIT['audit'])
    ensure('global',stems,TRAIN);ensure('tile',stems,TRAIN)
    bb=np.zeros(4,np.float64);cc=np.zeros(4,np.float64)
    per=[]
    for s in tqdm(stems,desc='Historical audit, diagnostic only'):
        d=prepare(s,label=False)
        p=action_probabilities(info['models'],d)
        rles,ledger=policy_predict(d,p,info['policy'])
        old=pooled_counts(s,[to_rle(q['counts']) for q in d['base']])
        new=pooled_counts(s,rles)
        bb+=old;cc+=new
        per.append({'stem':s,'baseline_pq':pq_score(old),'candidate_pq':pq_score(new),
                    'delta_pq':pq_score(new)-pq_score(old),**ledger})
    pd.DataFrame(per).to_csv(OUT/'diagnostics'/'historical_audit_per_image.csv',index=False)
    return {'ran':True,'baseline_pq':pq_score(bb),'candidate_pq':pq_score(cc),
            'delta':pq_score(cc)-pq_score(bb),'baseline_counts':pq_summary(bb),
            'candidate_counts':pq_summary(cc),
            'note':'Historical audit was already inspected; diagnostic only.'}

if ACTIONS:
    AUDIT_RESULT=run_audit(ACTIONS)
    (OUT/'audit_report.json').write_text(json.dumps(AUDIT_RESULT,indent=2))
    print('Historical audit:',AUDIT_RESULT)

## 22 · Strict promotion decision

**Every gate is mandatory**: historical recovery (`180/180` and `.995`), grouped OOF gain `+0.015`, both folds positive, previously inspected audit Δ ≥ −.005, official evaluator parity verified against an actual reference function, and no hidden test labels. We do **not** use public leaderboard score to tune the action thresholds.

In [ ]:
SELECTION=None
if ACTIONS:
    gates={'recovered_180':bool(RECOVERY['recovered']),
           'recovery_agreement':bool(RECOVERY['agreement']>=.995),
           'oof_gain':bool(ACTIONS['gain']>=MIN_GAIN_TO_PROMOTE),
           'fold0_positive':bool(ACTIONS['fold_gains'][0]>0),
           'fold1_positive':bool(ACTIONS['fold_gains'][1]>0),
           'audit_regression':bool(AUDIT_RESULT['ran'] and AUDIT_RESULT['delta']>=AUDIT_MIN_DELTA),
           'organizer_pq_parity_confirmed':bool(CONFIRMED_OFFICIAL_PQ_PARITY and OFFICIAL_PARITY_OK),
           'test_labels_used':False}
    promote=all(v for k,v in gates.items() if k!='test_labels_used')
    SELECTION={'promote':bool(promote),'gates':gates,'policy':ACTIONS['policy'],
               'baseline_validation_pq':ACTIONS['baseline'],
               'candidate_oof_pq':ACTIONS['candidate'],
               'absolute_gain':ACTIONS['gain'],'fold_gains':ACTIONS['fold_gains'],
               'audit':AUDIT_RESULT,'recovery':RECOVERY,
               'official_evaluator_parity_verified':bool(CONFIRMED_OFFICIAL_PQ_PARITY and OFFICIAL_PARITY_OK),
               'action_signature':ACTION_SIGNATURE,
               'test_labels_used':False,'public_lb_used_for_selection':False,
               'frozen_sha256':BASE_SHA,'checkpoint_sha256':MODEL_SHA,
               'split_sha256':SPLIT_SHA}
    ACTION_SELECTION_PATH.write_text(json.dumps(SELECTION,indent=2,default=str))
    print('PROMOTION:',promote,'gates:',gates)

## 23 · Correct instance-ID CSV export and automatic exact-0.34 rollback

`SUBMIT` uses only official H-alpha test images for live predictions, plus the pretrained action models from the validated `ACTIONS` stage. If anything fails, copy the frozen CSV **byte-for-byte** with SHA verification. If `ACTIONS` was run previously, attach its output as a Kaggle dataset.

In [ ]:
def get_previous_selection():
    if SELECTION is not None:return SELECTION
    for root in [OUT,PREVIOUS_V13_ROOT]:
        if root is None:continue
        path=root/'selection.json'
        if path.is_file():
            obj=json.loads(path.read_text())
            assert obj['frozen_sha256']==BASE_SHA and obj['checkpoint_sha256']==MODEL_SHA
            assert obj['split_sha256']==SPLIT_SHA
            assert obj.get('action_signature')==ACTION_SIGNATURE, 'Action features/proposals changed since validation.'
            return obj
    return None

def get_final_action_models():
    if ACTIONS:return ACTIONS['models']
    models={}
    for action in ('delete','add','merge'):
        candidates=[root/'models'/f'{action}_final.joblib' for root in [OUT,PREVIOUS_V13_ROOT] if root]
        path=next((p for p in candidates if p.is_file()),None)
        if path is None:raise FileNotFoundError('Missing model '+action+'; run ACTIONS first.')
        models[action]=joblib.load(path)
    return models

def export_submission():
    dest=Path('/kaggle/working/submission.csv')
    selection=get_previous_selection()
    # Do not trust stale selected JSON or environment changes; all provenance enforced.
    allowed=bool(selection and selection.get('promote') and
                 selection.get('recovery',{}).get('recovered') and
                 selection.get('official_evaluator_parity_verified') and
                 selection.get('test_labels_used') is False and
                 all(bool(v) for k,v in selection.get('gates',{}).items() if k!='test_labels_used') and
                 selection.get('action_signature')==ACTION_SIGNATURE and
                 RECOVERY['recovered'] and sha256(FROZEN)==BASE_SHA)
    if not allowed:
        shutil.copy2(FROZEN,dest)
        assert sha256(dest)==BASE_SHA
        print('FROZEN ROLLBACK: 0.34 original CSV copied byte-for-byte (no promoted candidate).')
        return {'promoted':False,'submission_sha256':sha256(dest),'output':str(dest)}
    models=get_final_action_models()
    ensure('global',sorted(TEST),TEST);ensure('tile',sorted(TEST),TEST)
    rows=[];ledger=[];comparison=[]
    for stem in tqdm(sorted(TEST),desc='Validated live test inference'):
        data=prepare(stem,label=False)
        probs=action_probabilities(models,data)
        rles,edits=policy_predict(data,probs,selection['policy'])
        assert masks_disjoint(rles),stem
        comparison.append({'stem':stem,'frozen_count':len(FROZEN_RLES.get(stem,[])),
                           'promoted_count':len(rles)})
        ledger.append({'stem':stem,**edits})
        for j,r in enumerate(rles,1):
            assert float(coco.area(r))>0
            rows.append((f'{stem}_{j}',to_ascii(r)))
    frame=pd.DataFrame(rows,columns=['filament_id','segmentation_rle'])
    assert not frame.empty and frame['filament_id'].is_unique
    assert frame['segmentation_rle'].astype(bool).all()
    for stem,group in frame.groupby(frame.filament_id.str.rsplit('_',n=1).str[0]):
        assert stem in TEST
        indices=sorted(int(v.rpartition('_')[-1]) for v in group['filament_id'])
        assert indices==list(range(1,len(indices)+1))
    frame.to_csv(dest,index=False)
    pd.DataFrame(ledger).to_csv(OUT/'delta_ledger.csv',index=False)
    pd.DataFrame(comparison).to_csv(OUT/'diagnostics'/'submission_vs_frozen.csv',index=False)
    print('PROMOTED NEW CANDIDATE:',dest,'rows:',len(frame))
    return {'promoted':True,'submission_sha256':sha256(dest),
            'output':str(dest),'rows':len(frame)}

REPORT=None
if MODE in ('SUBMIT','ALL'):
    REPORT=export_submission()
    (OUT/'submission_report.json').write_text(json.dumps(REPORT,indent=2))

## 24 · Manifest, output inventory and next steps

Outputs include baseline recovery tables, grouped action models, per-action ΔPQ tables, eight-policy validation, diagnostic ablations, audit, final selection and a reproducibility manifest. If `MODE='RECOVER'`, inspect `recovery/baseline_recovery.json` first. **A failed recovery does not certify a new score.**

In [ ]:
run_manifest={'version':'V13 SOURCE-PRESERVING RECONSTRUCTION',
 'original_v7_notebook_sha256':'33b3ff8d118be547096f6146973eb8e618d1eff6e0f60cefe5e5ff4db01dcffc',
 'frozen_baseline_sha256':sha256(FROZEN),'checkpoint_sha256':sha256(CKPT),
 'split_manifest_sha256':sha256(SPLIT_FILE),'official_train_annotation_sha256':sha256(ANNOT_PATH),
 'device':str(DEVICE),'ultralytics':ultralytics.__version__,
 'torch':torch.__version__,'opencv':cv2.__version__,
 'seed':SEED,'global_imgsz':GLOBAL_IMGSZ,'tile_imgsz':TILE_IMGSZ,
 'tile_side':TILE_SIDE,'raw_conf':RAW_CONF,'raw_iou':RAW_IOU,
 'recovery':RECOVERY,'selection':SELECTION,
 'historical_audit_is_pristine':False,'test_labels_used':False,
 'public_lb_feedback_used_for_selection':False}
(OUT/'reproducibility_manifest.json').write_text(json.dumps(run_manifest,indent=2,default=str))
(OUT/'pip_freeze.txt').write_text(subprocess.check_output(
    [sys.executable,'-m','pip','freeze'],text=True))
print('V13 run completed:',MODE,'Outputs:',OUT)
print('Baseline recovered?',RECOVERY['recovered'])
if SELECTION:print('Promotion:',SELECTION['promote'],'gain:',SELECTION['absolute_gain'])
if REPORT:print('Submission result:',REPORT)